# Agentic RAG Chatbot - "Attention Is All You Need"

This notebook demonsrates an Agentic RAG (Retrieval-Augmented Generation) pipeline built with LangGraph. The system can autonomously decide wether to reactive information evaluate retrieved documents for relevance, rewrite queries when results are poor, and check its own answers for hallucinations.

**Data source:** [Attention Is All You Need](https://arxiv.org/pdf/1706.03762) (Vaswani et al., 2017)

**Key patterns implemented:**
- Adaptive routing (retrieval vs. direct answer)
- Corrective RAG (document grading + query rewriting)
- Self-RAG (hallucination check + answer grading)

## 1. Setup and dependencies

Installing the required packages. Key components:
- **LangGraph** - state machine freamwork for the agentic pipeline
- **ChromaDB** - vector store for document embeddings
- **sentence-transformers** - multilingial embedding model
- **transforms + bitsandbytes** - local LLM inference with 4-bit quantization
- **PyMuPDF** - PDF text extraction

In [1]:
!pip install -q langgraph langchain langchain-core langchain-text-splitters langchain-community sentence-transformers chromadb pymupdf transformers bitsandbytes accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 35.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 62.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 68.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 9.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 73.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 43.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 68.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 180.2/180.2 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.0/69.0 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 k

## 2. Document loading

Downloading the original Transformer paper from arXiv and extracting raw text using PyMuPDF. This serves as our knowledge base - a single, well known paper is sufficient to demonstratte the full pipeline. The architecture is designed to scale to hundreds of documents without modifications.

In [2]:
!wget -q -O attention.pdf https://arxiv.org/pdf/1706.03762

import fitz
doc = fitz.open("attention.pdf")

print(f"Total pages: {len(doc)}")
print(f"First 500 character of the first page:\n{doc[0].get_text()[:500]}")

Total pages: 15
First 500 character of the first page:
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Parmar∗
Google Research
nikip@google.com
Jakob Uszkoreit∗
Google Research
usz@google.com
Llion Jones∗
Google Research
llion@google.com
Aidan N. Gomez∗†
University of Toronto
aidan@cs.toronto.edu
Łukasz K


##3 Document chunking

Splitting the paper into semantically meaningful chunks. The strategy is combines section-based splitting (preversing local boundries like "3.2.1 Scaled Dot-Product Attention") with recursive character splitting for sections that exceed the token limit. Each chunk carries metadata (section id, title) for source attribution in generated answers.

In [3]:
import fitz
import re

# Extract full text from PDF
doc = fitz.open("attention.pdf")
full_text = ""
for page in doc:
  text = page.get_text()
  text = re.sub(r'\n\s*\d+\s*$', '\n', text)
  full_text += text

# Cleaning
full_text = re.sub(r'arXiv:\d+\.\d+v\d+\s*\[.*?\]\s*\d+\s+\w+\s+\d{4}', '', full_text)
full_text = re.sub(r'\n{3,}', '\n\n', full_text)

# Split into sections based on heading pattern
section_pattern = re.compile(r'\n\s*(\d{1,2}(?:\.\d{1,2})*)\s+([A-Z][^\n]+)(?=\n)')
matches = list(section_pattern.finditer(full_text))

matches = [m for m in matches if not re.match(
    r'(Figure|Table)', m.group(2)
) and int(m.group(1).split('.')[0]) <= 7]

sections = []
for i, match in enumerate(matches):
  section_id = match.group(1)
  section_title = match.group(2).strip()
  start = match.end()
  end = matches[i +1 ].start() if i < len(matches) - 1 else len(full_text)
  content = full_text[start:end].strip()

  sections.append({
      "section_id": section_id,
      "section_title": section_title,
      "content": content,
  })

# Add abstract manually
abstract_end = matches[0].start() if matches else len(full_text)
abstract_text = full_text[:abstract_end]
abstract_match = re.search(r'Abstract\n(.*?)(?=\n\d)', abstract_text, re.DOTALL)
if abstract_match:
  sections.insert(0,{
      "section_id": "Abs",
      "section_title": "Abstract",
      "content": abstract_match.group(1).strip(),
  })

# Add References manually
if sections:
  last_section = sections[-1]
  tail_text = last_section["content"]

  ref_match = re.search(r'\n\s*References\s*\n', tail_text)
  vis_match = re.search(r'\n\s*Attention Visualizations\s*\n', tail_text)

  if ref_match:
    last_section["content"] = tail_text[:ref_match.start()].strip()

    ref_start = ref_match.end()
    ref_end = vis_match.start() if vis_match else len(tail_text)

    sections.append({
        "section_id": "Ref",
        "section_title": "References",
        "content": tail_text[ref_start:ref_end].strip(),
    })

  if vis_match:
    vis_start = vis_match.end()
    sections.append({
        "section_id": "Vis",
        "section_title": "Attention Visualizations",
        "content": tail_text[vis_start:].strip(),
    })

print(f"Found {len(sections)} sections:\n")
for s in sections:
  print(f"[{s['section_id']}] {s['section_title']} ({len(s['content'])} chars)")


Found 25 sections:

[Abs] Abstract (2127 chars)
[1] Introduction (1905 chars)
[2] Background (1815 chars)
[3] Model Architecture (744 chars)
[3.1] Encoder and Decoder Stacks (1316 chars)
[3.2] Attention (520 chars)
[3.2.1] Scaled Dot-Product Attention (1497 chars)
[3.2.2] Multi-Head Attention (1419 chars)
[3.2.3] Applications of Attention in our Model (1169 chars)
[3.3] Position-wise Feed-Forward Networks (615 chars)
[3.4] Embeddings and Softmax (1020 chars)
[3.5] Positional Encoding (1386 chars)
[4] Why Self-Attention (3179 chars)
[5] Training (58 chars)
[5.1] Training Data and Batching (591 chars)
[5.2] Hardware and Schedule (398 chars)
[5.3] Optimizer (447 chars)
[5.4] Regularization (1242 chars)
[6] Results (0 chars)
[6.1] Machine Translation (1669 chars)
[6.2] Model Variations (2117 chars)
[6.3] English Constituency Parsing (2397 chars)
[7] Conclusion (1216 chars)
[Ref] References (6994 chars)
[Vis] Attention Visualizations (2318 chars)


## 4. Chunk splitting

Sections that ewxceed 1000 characters are further split using LangChain's RecursiveCharacterTextSplitter with 200-character overlap to preserve context across chunk boundaries. Short sections remain intact. Each chunk inherits the original section metadata (section id, tilte, source)

In [11]:
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap  = 200,
    length_function = len,
    separators = [r"\n\n", r"\n", r"(?<=\. )", " ", ""]
)

documents = []

for sec in sections:
  if not sec["content"].strip():
    continue

  chunks = text_splitter.create_documents(
      texts = [sec["content"]],
      metadatas = [{
          "source": "Attention Is All You Need",
          "section_id": sec["section_id"],
          "section_title": sec["section_title"],
      }]
  )

  documents.extend(chunks)

print(f"Number of original sections: {len(sections)}")
print(f"Number of chunks: {len(documents)}\n")

test_idx = 15 if len(documents) > 15 else 0
print("Random chunk")
print(f"Contents:\n{documents[test_idx].page_content}")
print(f"Metadata:\n{documents[test_idx].metadata}")


Number of original sections: 25
Number of chunks: 56

Random chunk
Contents:
Instead of performing a single attention function with dmodel-dimensional keys, values and queries,
we found it beneficial to linearly project the queries, keys and values h times with different, learned
linear projections to dk, dk and dv dimensions, respectively. On each of these projected versions of
queries, keys and values we then perform the attention function in parallel, yielding dv-dimensional
4To illustrate why the dot products get large, assume that the components of q and k are independent random
variables with mean 0 and variance 1. Then their dot product, q · k = Pdk
i=1 qiki, has mean 0 and variance dk.
output values. These are concatenated and once again projected, resulting in the final values, as
depicted in Figure 2.
Multi-head attention allows the model to jointly attend to information from different representation
subspaces at different positions. With a single attention head, averaging in

##